# Preprocessing raw data

## Cell 1 – Project root and relative data paths

This cell defines the core folder structure for the project:

- `PROJECT_ROOT` is set to the main *Bachelorarbeit* directory (specified once as an absolute path),
- all other paths (e.g. `AnalysisScripts`, `Data/Raw`, `Data/Processed`) are defined **relative** to `PROJECT_ROOT`,
- (the `Data/Processed` folder is created if it does not exist)
- a filename pattern (`NN_ET_Data_YYYY-MM-DD.csv`) is defined to identify valid eye-tracking CSV files.

This setup keeps the analysis portable within the Bachelor thesis project while avoiding hard-coded system-specific paths in the later code.


In [1]:
import re
from pathlib import Path
import pandas as pd

from pprint import pprint

# ===================== CONFIG & PATHS =====================


PROJECT_ROOT = Path("/Users/luise/Library/CloudStorage/OneDrive-Persönlich/Bachelorarbeit")

# Ab hier NUR NOCH relative Pfade:
SCRIPT_DIR         = PROJECT_ROOT / "AnalysisScripts"
DATA_RAW_DIR       = PROJECT_ROOT / "Data" / "Raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"

# Ordner für verarbeitete Daten anlegen (falls nicht existent)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:       ", PROJECT_ROOT)
print("SCRIPT_DIR:         ", SCRIPT_DIR)
print("DATA_RAW_DIR:       ", DATA_RAW_DIR)
print("DATA_PROCESSED_DIR: ", DATA_PROCESSED_DIR)

# Allowed file name pattern: NN_ET_Data_YYYY-MM-DD.csv
FILENAME_PATTERN = re.compile(
    r"^(?P<uid>\d{2})_ET_Data_(?P<date>\d{4}-\d{2}-\d{2})\.csv$"
)


PROJECT_ROOT:        /Users/luise/Library/CloudStorage/OneDrive-Persönlich/Bachelorarbeit
SCRIPT_DIR:          /Users/luise/Library/CloudStorage/OneDrive-Persönlich/Bachelorarbeit/AnalysisScripts
DATA_RAW_DIR:        /Users/luise/Library/CloudStorage/OneDrive-Persönlich/Bachelorarbeit/Data/Raw
DATA_PROCESSED_DIR:  /Users/luise/Library/CloudStorage/OneDrive-Persönlich/Bachelorarbeit/Data/Processed


## Cell 2 – Helper functions for model validity, AF/Complexity, and condition mapping

This cell defines helper functions used to enrich the raw eye-tracking samples:

- `clean_str`: normalizes strings and removes NaN values,
- `is_valid_model`: identifies valid models (non-empty names that do not start with “tm”),
- `infer_AF`: derives the Abstract/Familiar category from the final character of the model name,
- `infer_complexity`: extracts the model’s complexity label (C1/C2/C3) from its name,
- `build_condition_map_for_uid`: implements the 6-item logic for assigning conditions
  (“constant”, “0.7s delay”, “2.0s delay”) based on the order in which each valid model first appears for a given participant.

These functions are reused across this preprocessing notebook and later analysis notebooks.


In [2]:
# ===================== HELPER FUNCTIONS =====================

def clean_str(x):
    """Convert NaN to empty string and strip whitespace."""
    return "" if pd.isna(x) else str(x).strip()

def is_valid_model(name: str) -> bool:
    """
    Return True only for real model names like 'C1M2A', 'C2M3F', etc.
    Rules:
      - value must not be NaN or empty,
      - must NOT start with 'tm' (TM models are excluded),
      - must match the pattern C[1-3]M<digit(s)><A/F>.
        Examples of valid names:
          C1M1A, C1M2F, C2M10A, C3M4F.
    All other values (including NaN and TM-prefixed names) are NOT valid.
    """
    n = clean_str(name)
    if not n:
        return False  # NaN oder leer → kein valid model
    if n.lower().startswith("tm"):
        return False  # TM-Modelle sind immer ausgeschlossen

    # Regex für "C1M2A" / "C2M10F" / "C3M4A" etc.
    pattern = re.compile(r"^C[123]M\d+[AF]$", flags=re.IGNORECASE)
    return bool(pattern.match(n))



def infer_AF(model_name: str) -> str:
    """
    Infer AF category from the LAST character of the model name.
      - 'A' -> 'Abstract'
      - 'F' -> 'Familiar'
      - otherwise -> 'Unknown'
    Example: 'C1M2A' -> 'Abstract'.
    """
    n = clean_str(model_name)
    if not n:
        return "Unknown"
    last = n[-1].upper()
    if last == "A":
        return "Abstract"
    if last == "F":
        return "Familiar"
    return "Unknown"


def infer_complexity(model_name: str) -> str:
    """
    Infer complexity level (C1, C2, C3) from the first 'C[1-3]' pattern.
    Example: 'C1M2A' -> 'C1', 'C2M3F' -> 'C2'.
    Returns 'NA' if no C1/C2/C3 pattern is found.
    """
    n = clean_str(model_name)
    m = re.search(r"(C[123])", n, flags=re.IGNORECASE)
    return m.group(1).upper() if m else "NA"


def build_condition_map_for_uid(df_uid: pd.DataFrame) -> dict:
    """
    Build a condition mapping for a single participant (UID) from model_name to condition label,
    based on the first time each NON-TM model appears.

    6-item logic:
      models  1–6   -> 'constant'
      models  7–12  -> '0.7s delay'
      models 13–18  -> '2.0s delay'
      all others    -> 'NA'
    """
    df_valid = df_uid.copy()
    df_valid["model_name"] = df_valid["model_name"].astype(str).str.strip()
    df_valid = df_valid[df_valid["model_name"].apply(is_valid_model)].copy()

    if "raw_timestamp" not in df_valid.columns:
        return {}

    df_valid["raw_timestamp"] = pd.to_numeric(df_valid["raw_timestamp"], errors="coerce")
    df_valid = df_valid.dropna(subset=["raw_timestamp"])
    if df_valid.empty:
        return {}

    first_seen = (
        df_valid.groupby("model_name", as_index=False)["raw_timestamp"].min()
                .rename(columns={"raw_timestamp": "first_seen_ts"})
                .sort_values("first_seen_ts")
    )

    order = first_seen["model_name"].tolist()
    cond_map = {}
    for idx, m_name in enumerate(order):
        if idx < 6:
            cond_map[m_name] = "constant"
        elif idx < 12:
            cond_map[m_name] = "0.7s delay"
        elif idx < 18:
            cond_map[m_name] = "2.0s delay"
        else:
            cond_map[m_name] = "NA"
    return cond_map


## Cell 3 – Load raw CSV files and build the combined `raw_all` dataframe

This cell loads all eye-tracking CSV files from `Data/Raw` whose filenames match the pattern
`NN_ET_Data_YYYY-MM-DD.csv`. For each valid file, it:

- reads the raw data into a dataframe,
- attaches metadata derived from the filename (`UID`, `recording_date`),
- stores the original filename in the column `source_file`.

All individual dataframes are concatenated into a single combined dataframe `raw_all`, which forms the basis for all subsequent preprocessing and enrichment steps.


In [3]:
# ===================== LOAD RAW CSVs & BUILD raw_all =====================

all_csv_paths = [p for p in DATA_RAW_DIR.glob("*.csv") if p.is_file()]

valid_files_meta = []
for p in all_csv_paths:
    m = FILENAME_PATTERN.match(p.name)
    if m:
        valid_files_meta.append((p, m.groupdict()))

if not valid_files_meta:
    raise FileNotFoundError(
        f"No CSVs in the format 'NN_ET_Data_YYYY-MM-DD.csv' found in {DATA_RAW_DIR}"
)

raw_dfs = []
for path, meta in valid_files_meta:
    df = pd.read_csv(path, low_memory=False)
    df["UID"] = meta["uid"]             # e.g. "01"
    df["recording_date"] = meta["date"] # e.g. "2025-08-08"
    df["source_file"] = path.name       # original filename
    raw_dfs.append(df)

raw_all = pd.concat(raw_dfs, ignore_index=True)

print("raw_all shape:", raw_all.shape)
raw_all.head()


raw_all shape: (4680329, 53)


,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,model_name,is_building_model,hit_obj_name,calibration_state,calibration_attempts,left_eye_calibration_quality,right_eye_calibration_quality,UID,recording_date,source_file
0,1000001256118443200,1755158559287,92.15744,2.000000,238330,0.0,Valid,-0.004600,-0.295994,0.955179,...,NaN,False,Wall_Front,WaitingForUser,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv
1,1000001276516059000,1755158580033,112.90440,0.125776,242408,0.0,Valid,0.013418,-0.067125,0.997654,...,NaN,False,Wall_Front,CheckingQuality,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv
2,1000001276826101900,1755158580033,112.90440,0.070757,242470,0.0,Valid,0.041925,0.066787,0.996886,...,NaN,False,Wall_Front,CheckingQuality,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv
3,1000001276831096400,1755158580033,112.90440,0.121814,242471,0.0,Valid,0.070052,0.124852,0.989699,...,NaN,False,Wall_Front,CheckingQuality,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv
4,1000001276836097200,1755158580033,112.90440,0.274620,242472,0.0,Valid,0.092422,0.147669,0.984709,...,NaN,False,Wall_Front,CheckingQuality,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv


In [18]:
raw_all["hit_obj_name"].unique()



array(['Wall_Front', 'ImageFixationCross', 'Floor', 'work_desk_base.001',
       'ValidationButtonRight', 'Wall_Left', 'ValidationButtonLeft',
       'PlaneLeft', 'PlaneRight', 'Lego_Board_10x10', 'stud_top_45',
       'stud_top_35', 'stud_top_25', 'stud_top_24', 'stud_top_12',
       'stud_top_11', 'stud_top_10', 'NextItemButton', 'Model_plate',
       'stud_top_39', 'stud_top_38', 'stud_top_27', 'stud_top_36',
       'stud_top_46', 'stud_top_44', 'stud_top_34', 'stud_top_54',
       'stud_top_85', 'stud_top_74', 'stud_top_64', 'stud_top_14',
       'stud_top_31', 'stud_top_41', 'stud_top_50', 'stud_top_60',
       'stud_top_61', 'stud_top_62', 'stud_top_76', 'stud_top_78',
       'stud_top_59', 'stud_top_68', 'stud_top_67', 'stud_top_77',
       'stud_top_86', 'Wall_Cut', 'Lego_1x3_Pink (1)', 'Lego_2x2_Orange',
       'Lego_4x2_Pink_Dis', 'stud_top_0', 'stud_top_3', 'Lego_4x2_Orange',
       'stud_top_4', 'stud_top_7', 'stud_top_6', 'stud_top_5',
       'stud_top_1', 'Lego_3x2_Orange

## Cell 4 – Enrich `raw_all` with participant IDs and model-level descriptors

This cell augments the combined raw dataframe with additional descriptive columns:

- cleaned `model_name` values,
- `is_valid_model`: a flag indicating whether a model is suitable for model-level analyses (non-empty, not starting with “tm”),
- `AF`: Abstract/Familiar classification inferred from the model name,
- `Complexity`: the model’s complexity level (C1/C2/C3),
- conversion of `raw_timestamp` to a numeric type in preparation for condition assignment.

These annotations allow downstream notebooks to work directly with the enriched samples without repeatedly re-computing this model-level logic.


In [4]:
# ===================== ENRICH raw_all WITH PARTICIPANT & MODEL INFO =====================

# Ensure model_name column exists
if "model_name" not in raw_all.columns:
    raw_all["model_name"] = ""

# Clean model_name
raw_all["model_name"] = raw_all["model_name"].astype(str).str.strip()

# Model validity (non-empty, not starting with 'tm')
raw_all["is_valid_model"] = raw_all["model_name"].apply(is_valid_model)

# AF category and complexity derived from model_name
raw_all["AF"] = raw_all["model_name"].apply(infer_AF)
raw_all["complexity"] = raw_all["model_name"].apply(infer_complexity)

# raw_timestamp is required for condition logic
if "raw_timestamp" not in raw_all.columns:
    raise ValueError("The column 'raw_timestamp' is missing in raw_all.")

raw_all["raw_timestamp"] = pd.to_numeric(raw_all["raw_timestamp"], errors="coerce")

print("Columns after enrichment:", raw_all.columns.tolist())
raw_all.head()


Columns after enrichment: ['gaze_capture_time', 'raw_timestamp', 'relative_to_unix_epoch_timestamp', 'focus_distance', 'frame_number', 'stability', 'status', 'gaze_forward_x', 'gaze_forward_y', 'gaze_forward_z', 'gaze_origin_x', 'gaze_origin_y', 'gaze_origin_z', 'left_forward_x', 'left_forward_y', 'left_forward_z', 'left_origin_x', 'left_origin_y', 'left_origin_z', 'left_status', 'left_pupil_diameter', 'left_iris_diameter', 'left_pupil_iris_ratio', 'left_eye_openness', 'right_forward_x', 'right_forward_y', 'right_forward_z', 'right_origin_x', 'right_origin_y', 'right_origin_z', 'right_status', 'right_pupil_diameter', 'right_iris_diameter', 'right_pupil_iris_ratio', 'right_eye_openness', 'inter_pupillary_distance', 'hmd_position_x', 'hmd_position_y', 'hmd_position_z', 'hmd_rotation_x', 'hmd_rotation_y', 'hmd_rotation_z', 'hmd_rotation_w', 'model_name', 'is_building_model', 'hit_obj_name', 'calibration_state', 'calibration_attempts', 'left_eye_calibration_quality', 'right_eye_calibration

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,calibration_state,calibration_attempts,left_eye_calibration_quality,right_eye_calibration_quality,UID,recording_date,source_file,is_valid_model,AF,complexity
0,1000001256118443200,1755158559287,92.15744,2.000000,238330,0.0,Valid,-0.004600,-0.295994,0.955179,...,WaitingForUser,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA
1,1000001276516059000,1755158580033,112.90440,0.125776,242408,0.0,Valid,0.013418,-0.067125,0.997654,...,CheckingQuality,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA
2,1000001276826101900,1755158580033,112.90440,0.070757,242470,0.0,Valid,0.041925,0.066787,0.996886,...,CheckingQuality,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA
3,1000001276831096400,1755158580033,112.90440,0.121814,242471,0.0,Valid,0.070052,0.124852,0.989699,...,CheckingQuality,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA
4,1000001276836097200,1755158580033,112.90440,0.274620,242472,0.0,Valid,0.092422,0.147669,0.984709,...,CheckingQuality,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA


## Cell 5 – Assign experimental conditions (6-item logic) per participant and model

This cell assigns an experimental condition to each valid model for every participant using the 6-item logic:

1. For each `UID`, valid (non-TM) models are identified.
2. The timestamp at which each model first appears (`raw_timestamp`) is determined.
3. Models are ordered by their first-seen time within each participant.
4. Conditions are assigned as follows:
   - models  1–6   → “constant”
   - models  7–12  → “0.7s delay”
   - models 13–18  → “2.0s delay”
   - all others    → “NA”.

The resulting condition labels are merged back into `raw_all`, so that each sample knows the condition of its associated model whenever applicable.


In [5]:
# ===================== ASSIGN CONDITION PER PARTICIPANT × MODEL =====================

records = []
for uid, df_uid in raw_all.groupby("UID"):
    cond_map_uid = build_condition_map_for_uid(df_uid)
    for model_name, cond in cond_map_uid.items():
        records.append({
            "UID": uid,
            "model_name": model_name,
            "condition": cond,
        })

cond_df = pd.DataFrame(records)

if cond_df.empty:
    print("⚠️ No conditions could be derived. All conditions will be set to 'NA'.")
    raw_all["condition"] = "NA"
else:
    raw_all = raw_all.merge(
        cond_df,
        on=["UID", "model_name"],
        how="left"
    )
    raw_all["condition"] = raw_all["condition"].fillna("NA")

print("Condition value counts:")
print(raw_all["condition"].value_counts(dropna=False))
raw_all.head()




Condition value counts:
condition
NA            2049111
2.0s delay     936929
0.7s delay     847298
constant       846991
Name: count, dtype: int64


,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,calibration_attempts,left_eye_calibration_quality,right_eye_calibration_quality,UID,recording_date,source_file,is_valid_model,AF,complexity,condition
0,1000001256118443200,1755158559287,92.15744,2.000000,238330,0.0,Valid,-0.004600,-0.295994,0.955179,...,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA,NA
1,1000001276516059000,1755158580033,112.90440,0.125776,242408,0.0,Valid,0.013418,-0.067125,0.997654,...,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA,NA
2,1000001276826101900,1755158580033,112.90440,0.070757,242470,0.0,Valid,0.041925,0.066787,0.996886,...,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA,NA
3,1000001276831096400,1755158580033,112.90440,0.121814,242471,0.0,Valid,0.070052,0.124852,0.989699,...,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA,NA
4,1000001276836097200,1755158580033,112.90440,0.274620,242472,0.0,Valid,0.092422,0.147669,0.984709,...,0,Unknown,Unknown,03,2025-08-14,03_ET_Data_2025-08-14.csv,False,Unknown,NA,NA


## Cell 6 – Save the enriched “corrected raw data” dataset

This cell saves the enriched combined dataframe (including `UID`, `participant_id`,
`recording_date`, `source_file`, `AF`, `Complexity`, `condition`, and `is_valid_model`) as:

`Data/Processed/corrected_raw_data.csv`

This “corrected raw data” file provides a stable, information-rich basis for subsequent analysis notebooks,
such as model-gaze/build-time analyses and world-object gaze overviews.


In [6]:
# ===================== SORT, PREVIEW, SAVE =====================

# Sort by participant and raw timestamp
raw_all_sorted = raw_all.sort_values(
    ["UID", "raw_timestamp"],
    ascending=[True, True]
).reset_index(drop=True)

print("Sorted shape:", raw_all_sorted.shape)

# Show first 100 rows so you can inspect all new columns
print("\nPreview of first 100 sorted rows:\n")
display(raw_all_sorted.head(100))

# Save to CSV
OUTPUT_CSV = DATA_PROCESSED_DIR / "corrected_raw_data.csv"
raw_all_sorted.to_csv(OUTPUT_CSV, index=False)

print(f"\n💾 Saved sorted 'corrected_raw_data' to:\n{OUTPUT_CSV}")


Sorted shape: (4680329, 57)

Preview of first 100 sorted rows:



,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,calibration_attempts,left_eye_calibration_quality,right_eye_calibration_quality,UID,recording_date,source_file,is_valid_model,AF,complexity,condition
0,1000002773811334400,1754658861466,5.185122,0.913979,257894,0.0,Valid,0.199822,-0.038434,0.979078,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA
1,1000002773816336100,1754658861467,5.185122,0.908018,257895,0.0,Valid,0.227848,-0.056482,0.972057,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA
2,1000002773821337800,1754658861467,5.185122,0.915157,257896,0.0,Valid,0.254252,-0.073261,0.964359,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA
3,1000002773826339500,1754658861467,5.185122,0.965864,257897,0.0,Valid,0.278826,-0.091104,0.956010,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA
4,1000002773831339600,1754658861467,5.185122,1.029571,257898,0.0,Valid,0.302282,-0.109190,0.946944,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1000002774286389800,1754658861473,5.185122,1.167749,257989,0.0,Valid,0.325093,-0.323355,0.888682,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA
96,1000002774291414600,1754658861473,5.185122,1.194077,257990,0.0,Valid,0.323222,-0.320255,0.890486,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA
97,1000002774296394100,1754658861473,5.185122,1.182748,257991,0.0,Valid,0.320705,-0.317847,0.892257,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA
98,1000002774301386300,1754658861488,5.214979,1.175651,257992,0.0,Valid,0.317145,-0.315870,0.894229,...,0,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA



💾 Saved sorted 'corrected_raw_data' to:
/Users/luise/Library/CloudStorage/OneDrive-Persönlich/Bachelorarbeit/Data/Processed/corrected_raw_data.csv


## Empirical Analysis of Desk-Run Lengths

Before finalizing the temporal label-correction heuristic, I conducted an empirical analysis of
Desk-run lengths (`work_desk_base.001`) in the raw data to determine an appropriate threshold
for correcting short Desk misclassifications inside LEGO interactions.

A *Desk run* is defined as a consecutive sequence of frames where the detected object is
`work_desk_base.001`. Short Desk runs that appear **within otherwise continuous LEGO
interactions** or **directly before the first LEGO detection** are likely caused by classifier
jitter rather than genuine user interaction with the desk. In contrast, longer Desk runs usually
represent meaningful contact with the surface and should remain untouched.

### Procedure
1. The dataset was sorted by participant (UID) and timestamp.
2. Consecutive Desk frames were grouped into Desk runs (`desk_run_id`).
3. For each Desk run, its run length (`desk_run_len`) was calculated.
4. The resulting distribution of Desk-run lengths was examined across the full dataset.
5. This distribution was used to assess whether short Desk runs (1–2 frames) dominate
   jitter-like behavior within LEGO contexts.




In [12]:
import pandas as pd
import numpy as np

# assume df is your raw (or pre-desk-heuristic) dataframe
df = raw_all_corr.copy()   # or whatever your variable is
df = df.sort_values(["UID", "raw_timestamp"])

def is_lego(name: str) -> bool:
    return str(name).strip().startswith("Lego_")

desk_name = "work_desk_base.001"

all_runs = []

for uid, seg in df.groupby("UID"):
    seg = seg.sort_values("raw_timestamp").copy()
    obj = seg["hit_obj_name"].astype(str)
    is_desk = obj.eq(desk_name)

    if not is_desk.any():
        continue

    # run IDs
    run_id = (is_desk != is_desk.shift(fill_value=False)).cumsum()
    seg["desk_run_id"] = np.where(is_desk, run_id, np.nan)

    # run length
    run_len = seg.groupby("desk_run_id")["hit_obj_name"].transform("size")
    seg["desk_run_len"] = np.where(is_desk, run_len, 0)

    # prev/next non-desk labels
    seg["prev_label"] = obj.where(~is_desk).ffill()
    seg["next_label"] = obj.where(~is_desk)[::-1].ffill()[::-1]

    # only desk rows
    desk_rows = seg[is_desk].copy()

    # mark LEGO-context as in the heuristic
    between_same_lego = (
        desk_rows["prev_label"].notna()
        & desk_rows["next_label"].notna()
        & (desk_rows["prev_label"] == desk_rows["next_label"])
        & desk_rows["prev_label"].apply(is_lego)
    )

    no_prev_obj = desk_rows["prev_label"].isna()
    next_is_lego = desk_rows["next_label"].apply(is_lego)
    before_first_lego = no_prev_obj & next_is_lego

    lego_context = between_same_lego | before_first_lego

    all_runs.append(desk_rows.loc[lego_context, ["desk_run_id", "desk_run_len"]])

# combine and look at run length distribution
runs = pd.concat(all_runs)
run_lengths = runs.drop_duplicates("desk_run_id")["desk_run_len"]

print(run_lengths.value_counts().sort_index())


desk_run_len
1.0      1668
2.0       297
3.0        86
4.0        75
5.0        42
         ... 
191.0       2
267.0       1
271.0       1
306.0       1
423.0       1
Name: count, Length: 128, dtype: int64


# Sensitivity check - nice table

In [21]:
# %%
import pandas as pd
import numpy as np

# assume df is your raw (or pre-desk-heuristic) dataframe
df = raw_all_corr.copy()
df = df.sort_values(["UID", "raw_timestamp"])

def is_lego(name: str) -> bool:
    return str(name).strip().startswith("Lego_")

desk_name = "work_desk_base.001"

all_runs = []

for uid, seg in df.groupby("UID"):
    seg = seg.sort_values("raw_timestamp").copy()
    obj = seg["hit_obj_name"].astype(str)
    is_desk = obj.eq(desk_name)

    if not is_desk.any():
        continue

    # run IDs
    run_id = (is_desk != is_desk.shift(fill_value=False)).cumsum()
    seg["desk_run_id"] = np.where(is_desk, run_id, np.nan)

    # run length
    run_len = seg.groupby("desk_run_id")["hit_obj_name"].transform("size")
    seg["desk_run_len"] = np.where(is_desk, run_len, 0)

    # prev/next non-desk labels
    seg["prev_label"] = obj.where(~is_desk).ffill()
    seg["next_label"] = obj.where(~is_desk)[::-1].ffill()[::-1]

    # only desk rows
    desk_rows = seg[is_desk].copy()

    # mark LEGO-context as in the heuristic
    between_same_lego = (
        desk_rows["prev_label"].notna()
        & desk_rows["next_label"].notna()
        & (desk_rows["prev_label"] == desk_rows["next_label"])
        & desk_rows["prev_label"].apply(is_lego)
    )

    no_prev_obj = desk_rows["prev_label"].isna()
    next_is_lego = desk_rows["next_label"].apply(is_lego)
    before_first_lego = no_prev_obj & next_is_lego

    lego_context = between_same_lego | before_first_lego

    all_runs.append(desk_rows.loc[lego_context, ["desk_run_id", "desk_run_len"]])

# combine and create table
if all_runs:
    runs = pd.concat(all_runs, ignore_index=True)
    run_lengths = runs.drop_duplicates("desk_run_id")["desk_run_len"]

    dist = (
        run_lengths.value_counts()
        .rename("count")
        .to_frame()
        .sort_index()
        .reset_index()
        .rename(columns={"index": "desk_run_len"})
    )
    dist["percent"] = (dist["count"] / dist["count"].sum() * 100).round(2)
else:
    dist = pd.DataFrame(columns=["desk_run_len", "count", "percent"])

dist


,desk_run_len,count,percent
0,1.0,1668,51.39
1,2.0,297,9.15
2,3.0,86,2.65
3,4.0,75,2.31
4,5.0,42,1.29
...,...,...,...
123,191.0,2,0.06
124,267.0,1,0.03
125,271.0,1,0.03
126,306.0,1,0.03


## Summary of Desk-Run Length Results

- Desk runs of length **1 frame** occurred 1668 times.
- Desk runs of length **2 frames** occurred 297 times.
- Combined, **1–2 frame Desk runs account for the overwhelming majority of short Desk detections**.
- Desk runs of length **3 frames or longer** occur far less often and increase gradually, indicating
  that these are more likely to represent genuine Desk interactions rather than jitter.
- Very long Desk runs (50–400+ frames) exist in the dataset, confirming that extended stable Desk
  contact is common and should not be modified by the heuristic.
- These findings suggest the threshold of **2 samples**:
  - It removes the vast majority of jitter.
  - It avoids incorrectly relabeling genuine Desk interactions.
  - It offers a safe and conservative balance between noise reduction and data integrity.

**Conclusion:**  
The empirical Desk-run distribution suggests
`max_desk_run_samples = 2` as an effective and conservative jitter-correction parameter.


# Creating the Heuristic that puts all Lego Instances together


### 1. Forward Pass (assign studs to the current LEGO)
As the data is processed from start to end, the heuristic keeps track of the most recently seen
LEGO object. All following `stud_top_*` and `stud_btm_*` detections are reassigned to this LEGO.
This removes fragmentations like:  
`Lego_X → stud_top → stud_btm → Lego_X`.

### 2. Backward Pass (fix studs before the first LEGO)
If stud detections appear *before* the first LEGO object in a segment, they are reassigned to the
next LEGO object. This handles noise where LEGO is detected slightly later than the studs.

### 3. Desk Jitter Correction (only when LEGO is clearly nearby)
Short Desk runs (≤ 2 frames) are corrected **only** when they occur in a reliable LEGO context:
- between the **same** LEGO object (e.g., `Lego_X → Desk → Desk → Lego_X`), or  
- at the beginning of the segment directly before the first LEGO.

All other Desk detections remain unchanged.  
This conservatively removes jitter while avoiding accidental relabeling of genuine Desk phases.

In [7]:
#finale Heuristik!!!

# ===================== LEGO-Heuristik (final) =====================

import pandas as pd
import numpy as np

# ----- Helper -------------------------------------------------------------

def is_lego(name: str) -> bool:
    """True, wenn ein Objekt ein LEGO-Objekt ist (Lego_...)."""
    return str(name).strip().startswith("Lego_")

def is_stud_top(name: str) -> bool:
    """True für stud_top_*."""
    return str(name).strip().startswith("stud_top_")

def is_stud_btm(name: str) -> bool:
    """True für stud_btm_*."""
    return str(name).strip().startswith("stud_btm_")


# ----- Desk -> LEGO nur bei 'LEGO in der Nähe' ----------------------------

def postprocess_workdesk_near_lego(
    df_segment: pd.DataFrame,
    obj_col: str = "hit_obj_name_corr",
    time_col: str = "raw_timestamp",
    desk_name: str = "work_desk_base.001",
    max_desk_run_samples: int = 2,
) -> pd.DataFrame:
    """
    Desk -> LEGO nur dann, wenn LEGO wirklich in der Nähe ist:

    - Desk-Run muss KURZ sein (<= max_desk_run_samples).
    - UND:
        a) Er liegt zwischen dem GLEICHEN LEGO-Objekt
           (z.B. Lego_4x2_Pink ... Desk ... Lego_4x2_Pink)
       ODER
        b) Er liegt am Anfang des Segments und direkt danach kommt LEGO
           (Desk ... Lego_4x2_Pink).

    - Alle anderen Desk-Runs bleiben Desk.
    """

    df = df_segment.sort_values(time_col).copy()

    # Falls korrigierte Spalte noch nicht existiert: vom Original kopieren
    if obj_col not in df.columns:
        df[obj_col] = df["hit_obj_name"]

    is_desk = df[obj_col].astype(str).eq(desk_name)
    if not is_desk.any():
        return df

    # Runs von Desk-Samples erkennen
    run_id = (is_desk != is_desk.shift(fill_value=False)).cumsum()
    df["desk_run_id"] = np.where(is_desk, run_id, np.nan)

    # Länge der Runs in Samples
    run_len = df.groupby("desk_run_id")[obj_col].transform("size")
    df["desk_run_len"] = np.where(is_desk, run_len, 0)

    # Kontext: vorheriges & nächstes NICHT-Desk-Label
    df["prev_label"] = df[obj_col].where(~is_desk).ffill()
    df["next_label"] = df[obj_col].where(~is_desk)[::-1].ffill()[::-1]

    short_run = is_desk & (df["desk_run_len"] <= max_desk_run_samples)

    # ---- Fall (a): kurzer Desk-Run zwischen gleichem LEGO ----
    between_same_lego = (
        df["prev_label"].notna()
        & df["next_label"].notna()
        & (df["prev_label"] == df["next_label"])
        & df["prev_label"].apply(is_lego)
    )
    jitter_between = short_run & between_same_lego
    df.loc[jitter_between, obj_col] = df.loc[jitter_between, "prev_label"]

    # ---- Fall (b): kurzer Desk-Run am Anfang vor erstem LEGO ----
    no_prev_obj = df["prev_label"].isna()
    next_is_lego = df["next_label"].apply(is_lego)
    jitter_before_first = short_run & no_prev_obj & next_is_lego
    df.loc[jitter_before_first, obj_col] = df.loc[jitter_before_first, "next_label"]

    # Aufräumen
    df = df.drop(columns=["desk_run_id", "desk_run_len", "prev_label", "next_label"])
    return df


# ----- Bidirektionale LEGO-Heuristik pro Segment --------------------------

def apply_heuristic_bidirectional(
    df_segment: pd.DataFrame,
    time_col: str = "raw_timestamp",
    desk_name: str = "work_desk_base.001",
    max_desk_run_samples: int = 2,
) -> pd.DataFrame:
    """
    Wendet die LEGO-Heuristik auf ein zeitlich sortiertes Segment (z.B. eine UID)
    an und gibt ein Segment mit 'hit_obj_name_corr' zurück.
    """

    df = df_segment.copy()
    df[time_col] = pd.to_numeric(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).sort_values(time_col)

    # ---------- Forward Pass: LEGO + studs ----------
    lego_mask = df["hit_obj_name"].apply(is_lego)
    df["lego_name"] = df["hit_obj_name"].where(lego_mask, pd.NA)
    df["prev_lego_owner"] = df["lego_name"].ffill()

    corrected = []
    for raw, prev_owner in zip(df["hit_obj_name"], df["prev_lego_owner"]):
        n = str(raw).strip()

        if is_lego(n):
            # LEGO bleibt LEGO
            corrected.append(n)

        elif is_stud_top(n) or is_stud_btm(n):
            # studs folgen dem zuletzt gesehenen LEGO (wenn vorhanden)
            if pd.notna(prev_owner):
                corrected.append(prev_owner)
            else:
                corrected.append(n)

        else:
            # Tisch, Floor, Walls etc. bleiben zunächst unverändert
            corrected.append(n)

    df["hit_obj_name_corr"] = corrected

    # ---------- Backward Pass: studs vor erstem LEGO ----------
    df["next_lego_owner"] = df["lego_name"][::-1].ffill()[::-1]

    mask_stud = df["hit_obj_name"].apply(
        lambda x: is_stud_top(x) or is_stud_btm(x)
    )
    no_prev = df["prev_lego_owner"].isna()
    has_next = df["next_lego_owner"].notna()

    backfill_mask = mask_stud & no_prev & has_next
    df.loc[backfill_mask, "hit_obj_name_corr"] = df.loc[
        backfill_mask, "next_lego_owner"
    ]

    # ---------- Desk-Jitter nur bei wirklichem LEGO-Kontext ----------
    df = postprocess_workdesk_near_lego(
        df,
        obj_col="hit_obj_name_corr",
        time_col=time_col,
        desk_name=desk_name,
        max_desk_run_samples=max_desk_run_samples,
    )

    # Aufräumen
    df = df.drop(columns=["lego_name", "prev_lego_owner", "next_lego_owner"])
    return df


# ----- Auf gesamten Datensatz anwenden ------------------------------------

def apply_heuristic_to_all(
    df: pd.DataFrame,
    uid_col: str = "UID",
    time_col: str = "raw_timestamp",
    desk_name: str = "work_desk_base.001",
    max_desk_run_samples: int = 2,
) -> pd.DataFrame:
    """
    Wendet die LEGO-Heuristik auf den gesamten Datensatz an,
    gruppiert nach Teilnehmer (uid_col).

    Gibt eine Kopie von df mit 'hit_obj_name_corr' zurück.
    """

    if uid_col not in df.columns:
        raise KeyError(f"UID column '{uid_col}' not found in dataframe.")

    if time_col not in df.columns:
        raise KeyError(f"Time column '{time_col}' not found in dataframe.")

    df_sorted = df.sort_values([uid_col, time_col]).copy()

    result = (
        df_sorted
        .groupby(uid_col, group_keys=False)
        .apply(
            lambda seg: apply_heuristic_bidirectional(
                seg,
                time_col=time_col,
                desk_name=desk_name,
                max_desk_run_samples=max_desk_run_samples,
            )
        )
    )

    return result


# Adding hit_obj_name_corr as column to raw_all


In [8]:
# LEGO-Heuristik auf gesamten Preprocess-DF anwenden
raw_all_corr = apply_heuristic_to_all(
    raw_all_sorted,
    uid_col="UID",
    time_col="raw_timestamp",
    desk_name="work_desk_base.001",
    max_desk_run_samples=2,   # ggf. an Samplingrate anpassen
)

print("✅ Heuristik angewendet.")
print("Spalte 'hit_obj_name_corr' vorhanden:", "hit_obj_name_corr" in raw_all_corr.columns)
print("Anzahl Zeilen:", len(raw_all_corr))

# kleiner Check
raw_all_corr


/var/folders/7k/2zmy2sns485034b8qp7j11440000gn/T/ipykernel_896/2369302033.py:189: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


✅ Heuristik angewendet.
Spalte 'hit_obj_name_corr' vorhanden: True
Anzahl Zeilen: 4680329


,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,left_eye_calibration_quality,right_eye_calibration_quality,UID,recording_date,source_file,is_valid_model,AF,complexity,condition,hit_obj_name_corr
0,1000002773811334400,1754658861466,5.185122,0.913979,257894,0.000000,Valid,0.199822,-0.038434,0.979078,...,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA,Floor
1,1000002773816336100,1754658861467,5.185122,0.908018,257895,0.000000,Valid,0.227848,-0.056482,0.972057,...,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA,Floor
2,1000002773821337800,1754658861467,5.185122,0.915157,257896,0.000000,Valid,0.254252,-0.073261,0.964359,...,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA,Floor
3,1000002773826339500,1754658861467,5.185122,0.965864,257897,0.000000,Valid,0.278826,-0.091104,0.956010,...,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA,Floor
4,1000002773831339600,1754658861467,5.185122,1.029571,257898,0.000000,Valid,0.302282,-0.109190,0.946944,...,Unknown,Unknown,01,2025-08-08,01_ET_Data_2025-08-08.csv,False,Unknown,NA,NA,Floor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4680324,1000004046721921400,1756802449893,2583.165000,0.333347,797593,0.744113,Valid,-0.029452,-0.292397,0.955844,...,High,High,10,2025-09-02,10_ET_Data_2025-09-02.csv,False,Unknown,NA,NA,work_desk_base.001
4680325,1000004046726922200,1756802449893,2583.165000,0.334417,797594,0.759231,Valid,-0.029293,-0.292608,0.955784,...,High,High,10,2025-09-02,10_ET_Data_2025-09-02.csv,False,Unknown,NA,NA,work_desk_base.001
4680326,1000004046731923000,1756802449904,2583.176000,0.334728,797595,0.766584,Valid,-0.029264,-0.292938,0.955683,...,High,High,10,2025-09-02,10_ET_Data_2025-09-02.csv,False,Unknown,NA,NA,work_desk_base.001
4680327,1000004046736920600,1756802449904,2583.176000,0.335252,797596,0.775193,Valid,-0.029070,-0.293248,0.955594,...,High,High,10,2025-09-02,10_ET_Data_2025-09-02.csv,False,Unknown,NA,NA,work_desk_base.001


# Save Final Preprocess - DF under the name: "preprocessed_raw_all"

In [9]:
# Finalen Preprocess-DF speichern
OUTPUT_ALL = DATA_PROCESSED_DIR / "preprocessed_raw_all.csv"
raw_all_corr.to_csv(OUTPUT_ALL, index=False)

print("✅ Full Preprocess-DF gespeichert unter:", OUTPUT_ALL)


✅ Full Preprocess-DF gespeichert unter: /Users/luise/Library/CloudStorage/OneDrive-Persönlich/Bachelorarbeit/Data/Processed/preprocessed_raw_all.csv


In [ ]:
extra_columns = ["UID", "recording_date", "source_file","","source_file","source_file","source_file","source_file",]

df[extra_columns].head()


# Last check if everything fits and is sorted!

In [10]:
import pandas as pd

# Lade die exportierte Preprocess-Datei
df = pd.read_csv(DATA_PROCESSED_DIR / "preprocessed_raw_all.csv")


/var/folders/7k/2zmy2sns485034b8qp7j11440000gn/T/ipykernel_896/747107032.py:4: DtypeWarning: Columns (55,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PROCESSED_DIR / "preprocessed_raw_all.csv")


# Checking if heuristic worked when added to DF (on Model: C2M4A)

In [11]:
import pandas as pd

# 1. Load the exported preprocess file
df = pd.read_csv(DATA_PROCESSED_DIR / "preprocessed_raw_all.csv")

# 2. Filter to model C2M4A and keep only the 3 columns
df_c2m4a = df.loc[
    df["model_name"] == "C2M4A",
    ["model_name", "hit_obj_name", "hit_obj_name_corr"]
]

# 3. Show the result (this is what Data Wrangler can hook into)
df_c2m4a  # or: display(df_c2m4a)


/var/folders/7k/2zmy2sns485034b8qp7j11440000gn/T/ipykernel_896/3296325200.py:4: DtypeWarning: Columns (55,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PROCESSED_DIR / "preprocessed_raw_all.csv")


,model_name,hit_obj_name,hit_obj_name_corr
254199,C2M4A,Wall_Front,Wall_Front
254200,C2M4A,Wall_Front,Wall_Front
254201,C2M4A,Wall_Front,Wall_Front
254202,C2M4A,Wall_Front,Wall_Front
254203,C2M4A,Wall_Front,Wall_Front
...,...,...,...
4428348,C2M4A,NextItemButton,NextItemButton
4428349,C2M4A,NextItemButton,NextItemButton
4428350,C2M4A,NextItemButton,NextItemButton
4428351,C2M4A,NextItemButton,NextItemButton
